In [4]:
from langchain.agents import create_agent
from langchain.tools import tool

@tool
def get_weathers(city: str) -> str:
    """Get weather for a given city."""
    return f"It's always sunny in {city}!"

agent = create_agent(
    model="ollama:llama3.2:3b",
    # tools=[get_weathers],
    system_prompt="You are a helpful assistant",
)

result = agent.invoke(
    {"messages": [{"role": "user", "content": "What's the climate like in San Francisco?"}]}
)

print(result["messages"][-2].content)


What's the climate like in San Francisco?


In [ ]:
# HOW TOOLS DIRECTLY BIND TO A MODEL MANUALLY - WORKING OF CREATE_AGENT
from langchain_ollama import ChatOllama
from langchain.tools import tool

@tool
def get_weather(city: str) -> str:
    """Get weather for a given city."""
    return f"It's always sunny in {city}!"

# Initialize raw model directly
model = ChatOllama(model="llama3.2:3b")

# Bind tool schema directly to the model
model_with_tools = model.bind_tools([get_weather])

response = model_with_tools.invoke("What's the weather in Boston?")
for tool_call in response.tool_calls:
    print(tool_call["name"], tool_call["args"])

get_weather {'city': 'Boston'}


In [12]:
# WRITE THE CODE TO CALL THE MODEL DIRECTLY WITHOUT USING CREATE_AGENT USING THE UNIVERSAL MODEL CONSTRUCTOR - INIT_CHAT_MODEL
# init_chat_model: A universal factory function in LangChain that instantiates chat models across different providers 
# using a standardized string format ("provider:model_name").
from langchain.chat_models import init_chat_model

model = init_chat_model(
    "ollama:llama3.2:3b",
    temperature=0.1,   # 0 = deterministic, 1 = creative
    timeout=300,
    max_tokens=1000,
)
# response = model.invoke("Why do parrots talk?")
# print(response.content)

# full = None  # None | AIMessageChunk
# for chunk in model.stream("What color is the sky?"):
#     full = chunk if full is None else full + chunk
#     print(full.text, end="", flush=True)

# The
# The sky
# The sky is
# The sky is typically
# The sky is typically blue
# ...

full = None  # None | AIMessageChunk
for chunk in model.stream("What color is the sky?"):
    full = chunk if full is None else full + chunk
    print(chunk.text, end="", flush=True) #Passing end="" replaces \n with an empty string, allowing each chunk to print horizontally on the exact same line.

# print(full.content_blocks)

The color of the sky can vary depending on the time of day and atmospheric conditions. Here are some common colors of the sky:

1. **Blue**: During the daytime, when the sun is overhead, the sky typically appears blue. This is because of a phenomenon called Rayleigh scattering, where shorter wavelengths of light (like blue and violet) are scattered more than longer wavelengths (like red and orange) by the tiny molecules of gases in the atmosphere.
2. **White or light gray**: At sunrise and sunset, the sky can take on hues of white or light gray, as the sun's rays pass through more of the Earth's atmosphere, scattering the shorter wavelengths of light.
3. **Pink or orange**: During sunrise and sunset, the sky can also appear pink or orange, due to the scattering of light by atmospheric particles and the presence of dust and water vapor.
4. **Gray or overcast**: On cloudy days, the sky can appear gray or overcast, as the clouds block the direct sunlight and scatter the light in all direc

In [ ]:
#invoke with exact 50 tokens which cuts off the response at 50 token exactly
from langchain.chat_models import init_chat_model

model = init_chat_model(
    "ollama:llama3.2:3b",
    temperature=0.1,   # 0 = deterministic, 1 = creative
    timeout=50,
    # max_tokens=200,
    num_predict = 50  # Ollama's native parameter for max completion tokens
)
response = model.invoke("What color is the sky?")
print(response.content)

The color of the sky can vary depending on the time of day and atmospheric conditions. Here are some common colors of the sky:

1. **Blue**: During the daytime, when the sun is overhead, the sky typically appears blue. This is because


In [15]:
from pydantic import BaseModel, Field
from langchain.chat_models import init_chat_model

class SingleSentenceResponse(BaseModel):
    """ A concise, single-sentence summary under 20 words without affecting the original meaning"""
    sentence: str
    
model = init_chat_model("ollama:llama3.2:3b", temperature=0.1)
structured_model = model.with_structured_output(SingleSentenceResponse)

response = structured_model.invoke("What is the name of the game with dinosour which pops in the browser with 'Press space to play' message?")
print(response.sentence)

The game you are referring to is likely 'Dino Daze' or 'Dino Run', but the most popular game with this description is 'Dino Daze'.


In [35]:
# Structured output at the model level
from pydantic import BaseModel, Field

model = init_chat_model(
    "ollama:llama3.2:3b",
    temperature=0.1,   # 0 = deterministic, 1 = creative
    timeout=300,
    max_tokens=1000,
)

class Movie(BaseModel):
    title: str = Field(description="The title of the movie")
    year: int
    rating: float
    director: str
    budget: str
    music: str
    release_date: str
    studio: str
    writer: str
    producer: str
    runtime: str
    box_office: str


# Bind schema and execute
model_with_structure = model.with_structured_output(Movie)
response = model_with_structure.invoke("Details about Inception")
print(response)
print(response.title)
print(response.year)
print(response.rating)
print(response.director)
print(response.budget)
print(response.music)
print(response.release_date)
print(response.studio)
print(response.writer)
print(response.producer)
print(response.runtime)
print(response.box_office)

title='Inception' year=2010 rating=8.8 director='Christopher Nolan' budget='>$160 million' music='Hans Zimmer' release_date='July 16, 2010' studio='Warner Bros.' writer='Christopher Nolan' producer='Christopher Nolan' runtime='148 minutes' box_office='$825.5 million worldwide'
Inception
2010
8.8
Christopher Nolan
>$160 million
Hans Zimmer
July 16, 2010
Warner Bros.
Christopher Nolan
Christopher Nolan
148 minutes
$825.5 million worldwide


In [24]:
# batch
model = init_chat_model(
    "ollama:llama3.2:3b",
    temperature=0.1,   # 0 = deterministic, 1 = creative
    timeout=300,
    max_tokens=1000,
)

responses = model.batch([
    "Why do parrots have colorful feathers?",
    "How do airplanes fly?",
    "What is quantum computing?"
])
for response in responses:
    print(response)

content='Parrots have colorful feathers for several reasons:\n\n1. **Communication and Social Behavior**: Bright colors play a crucial role in parrot communication and social behavior. They use their vibrant plumage to convey information about their identity, sex, age, and social status. Different colors can signal aggression, courtship, or warning calls.\n2. **Mating and Attraction**: Colorful feathers are essential for attracting a mate. Males use their bright plumage to showcase their health, strength, and genetic quality to potential partners. Females, on the other hand, may choose a mate based on the quality and color of his feathers.\n3. **Camouflage and Concealment**: While parrots are known for their bright colors, some species have evolved to blend in with their surroundings. For example, the green parrot has a greenish-gray plumage that helps it blend in with the foliage of its rainforest habitat.\n4. **Thermoregulation**: In some species, the color of the feathers can help r

In [2]:
from langchain.agents import create_agent
from langchain.tools import tool

agent = create_agent(
    model="ollama:llama3.2:3b",
    system_prompt="You are a helpful assistant",
)

for response in agent.batch_as_completed([
    "Why do parrots have colorful feathers?",
    "How do airplanes fly?",
    "What is quantum computing?"
]):
    print(response.text)

InvalidUpdateError: Expected dict, got Why do parrots have colorful feathers?
For troubleshooting, visit: https://docs.langchain.com/oss/python/langgraph/errors/INVALID_GRAPH_NODE_RETURN_VALUE